In [0]:
print("===================================")
print("CUSTOMER SEGMENTATION PROJECT")
print("===================================")

print("Platform: Databricks")
print("Language: Python")
print("Processing: PySpark")
print("Storage: Delta Lake")

print("Project setup successful!")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
# Project configuration

SEED = 20261022

CUSTOMER_COUNT = 1000

print("Random Seed:", SEED)
print("Number of Customers:", CUSTOMER_COUNT)

In [0]:
# Generate customer IDs

customers = spark.range(1, CUSTOMER_COUNT + 1)

customers = customers.withColumn(
    "customer_id",
    F.concat(
        F.lit("CUS-"),
        F.lpad(F.col("id").cast("string"), 6, "0")
    )
)



In [0]:
# Generate deterministic values

customers = customers.withColumn(
    "random_value",
    F.pmod(F.xxhash64("customer_id", F.lit(SEED)), F.lit(100000))
)

customers = customers.withColumn(
    "age",
    (F.col("random_value") % 48 + 18).cast("int")
)

customers = customers.withColumn(
    "gender",
    F.when(F.col("random_value") % 2 == 0, "Female")
     .otherwise("Male")
)

customers = customers.withColumn(
    "city",
    F.when(F.col("random_value") % 5 == 0, "Bangalore")
     .when(F.col("random_value") % 5 == 1, "Mumbai")
     .when(F.col("random_value") % 5 == 2, "Delhi")
     .when(F.col("random_value") % 5 == 3, "Hyderabad")
     .otherwise("Chennai")
)

customers = customers.withColumn(
    "acquisition_channel",
    F.when(F.col("random_value") % 4 == 0, "Website")
     .when(F.col("random_value") % 4 == 1, "Mobile App")
     .when(F.col("random_value") % 4 == 2, "Referral")
     .otherwise("Social Media")
)

customers = customers.withColumn(
    "signup_date",
    F.date_add(
        F.lit("2024-01-01").cast("date"),
        (F.col("random_value") % 1033).cast("int")
    )
)

customers = customers.withColumn(
    "account_status",
    F.lit("ACTIVE")
)



In [0]:
# Select final columns

customer_df = customers.select(
    "customer_id",
    "age",
    "gender",
    "city",
    "signup_date",
    "acquisition_channel",
    "account_status"
)

print("Customer data generated successfully!")


In [0]:
#Display the customer data
customer_df.show(10, truncate=False)

In [0]:
#Check the customer count
print("Total Customers:", customer_df.count())

customer_df.printSchema()

In [0]:
#Validate customer IDs
# Check duplicate customer IDs

duplicate_count = (
    customer_df.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate Customer IDs:", duplicate_count)

In [0]:
#Save customer data in Delta Lake
customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_master")

In [0]:
%sql
SELECT * FROM customer_master LIMIT 10;